In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from obspy import read
import matplotlib.cm as cm

import numpy as np
import obspy
from obspy import read, Trace, Stream
import scipy.fft

from matplotlib import colors
from scipy.signal import windows

# Path to the new MiniSEED file


# Read the MiniSEED file
st = read(path)

# Select only Z-components
#st = st.select(component="Z")

st_to_vel = st.copy()

st_to_vel.detrend("demean")
st_to_vel.taper(0.05, type="cosine")
st_to_vel.filter("bandpass", freqmin=0.1, freqmax=100)
st_to_vel.normalize()

# strain to vel

# Step 2: Extract the data (assuming each trace in the stream is a numpy array)
data = np.array([tr.data for tr in st_to_vel])  # Shape: (num_traces, time_points)

print(data.shape)

npts = data.shape[1]
ntrs = data.shape[0]
dt = st_to_vel[0].stats.delta
dx = 2

print(npts,ntrs,dt,dx)

hann=np.hanning(data.shape[1])
data=data*hann
# FK spectra calculation
f = np.fft.rfftfreq(npts, d=dt)
k = np.fft.fftfreq(ntrs, d=dx)
fk= np.fft.rfft2(data) 

print(k.shape)

print(k)


In [ ]:
# Plot the spectra
plt.figure(figsize=[8,6])
plt.imshow(np.abs(np.fft.fftshift(fk, axes=(0,))).T, extent=[min(k), max(k), min(f), max(f)],
            aspect='auto', cmap='plasma', interpolation=None, origin='lower', norm=colors.LogNorm())
fk_abs = np.abs(fk) / np.max(np.abs(fk))

h = plt.colorbar()
h.set_label('Amplitude Spectra  (rel. 1 $(\epsilon/s)^2$)')
plt.ylabel('frequency [Hz]', fontsize=18)
plt.xlabel('wavenumber [1/m]', fontsize=18)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()

In [ ]:
# Avoid division by zero

k = np.fft.fftshift(np.fft.fftfreq(ntrs, d=dx))

# Reshape k to match the shape of fk for broadcasting
k_safe = np.where(np.abs(k[:, None]) < 1e-3, np.inf, k[:, None])

# Now fk and k_safe are broadcastable
fk_divided = np.zeros_like(fk)  # Initialize with zeros
fk_divided = fk / k_safe  # Perform division

# Optional: Set values where k_safe is inf (previously zero in k) to zero
fk_divided[np.isinf(fk_divided)] = 0

data_time_domain =np.fft.irfft2(fk_divided, s=data.shape)                     # Inverse transform

strain_to_vel = np.diff(data_time_domain, axis=1) / dt

print(strain_to_vel.shape)



In [ ]:
new_traces = []
for i, trace_data in enumerate(strain_to_vel):
    # Only take the real part of the data (since MiniSEED cannot store complex data)
    real_trace_data = np.real(trace_data)
    
    # Create a new Trace for each processed signal
    tr = Trace(data=real_trace_data)
    tr.stats = st_to_vel[i].stats  # Copy the stats (e.g., network, station, location, etc.)
    new_traces.append(tr)

# Step 13: Create a new Stream object with the new traces
strain_to_vel = Stream(traces=new_traces)

print(strain_to_vel)

# Preprocess: Detrend, Taper, Bandpass Filter, and Normalize
strain_to_vel.detrend("demean")
strain_to_vel.taper(0.05, type="cosine")
strain_to_vel.filter("bandpass", freqmin=2, freqmax=30)

# Normalize each trace
for tr in strain_to_vel:
    tr.data = tr.data / np.max(np.abs(tr.data))

# Create the figure with a dark background
fig, ax = plt.subplots(figsize=(12, 6), dpi=300)
ax.set_facecolor("black")
fig.patch.set_facecolor("black")

# Select only every 200th seismogram for simplicity (you can adjust the step as needed)
indices_to_plot = range(0, len(strain_to_vel), 200)

# Time axis
t = np.linspace(0, strain_to_vel[0].stats.npts / strain_to_vel[0].stats.sampling_rate, strain_to_vel[0].stats.npts)

# Normalize station index to the color map
norm = plt.Normalize(vmin=0, vmax=len(strain_to_vel) - 1)
cmap = plt.cm.get_cmap("BuGn", len(strain_to_vel))

# Plot each trace with an offset and color from the colormap
offset = 1.5  # Vertical offset between traces
for i, idx in enumerate(indices_to_plot):
    tr = strain_to_vel[idx]
    color = cmap(norm(idx))  # Get the color based on station index
    ax.plot(t, tr.data + i * offset, color='white', lw=1)

# Formatting the plot
ax.set_xlabel("Time [s]", color="white", fontsize=12)
ax.set_ylabel("Station Index", color="white", fontsize=12)
ax.set_yticks(np.arange(len(indices_to_plot)) * offset)
ax.set_yticklabels([st[idx].stats.station for idx in indices_to_plot], color="white", fontsize=10)
ax.spines["top"].set_color("white")
ax.spines["bottom"].set_color("white")
ax.spines["left"].set_color("white")
ax.spines["right"].set_color("white")
ax.tick_params(colors="white")

# Title
ax.set_title("DAS Channels (strain to vel)", color="white", fontsize=14)

# Create a colorbar to map the color scale to station indices
#sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
#sm.set_array([])  # Set the array to an empty one to create the colorbar
#cbar = plt.colorbar(sm, ax=ax, label='DAS ID', location='right', fraction=0.02, pad=0.04)
#cbar.ax.tick_params(labelsize=8)

# Show the plot
plt.show()



#select channels 

#selected_stations_hybrid = ["0223", "1300", "3039", "5000", "7185"]  # Selected station IDs

# Create empty stream for hybrid data
subsampled_vel_stream = Stream()


e = strain_to_vel.copy()

stations_sf = sta_filt[0].stats.sampling_rate 

# Select only the traces corresponding to selected hybrid stations
for a in selected_stations_hybrid:
    b_new = e.select(station=str(a))
    c_new = b_new.resample(stations_sf)


    trace = c_new[0]
    trace.stats.channel = "HHE"  # Set to desired component name

    subsampled_vel_stream.append(trace)

    #trace_N = copy.deepcopy(c_new[0]) 
    #trace_E = copy.deepcopy(c_new[0]) 
    #trace_N.stats.channel = "HHE"
    #trace_E.stats.channel = "HHN"

    #subsampled_vel_stream.append(c_new[0])
    #subsampled_vel_stream += trace_N
    #subsampled_vel_stream += trace_E



### save the file 


subsampled_vel_stream.normalize()

# Combine with filtered seismometer data
subsampled_filtered_stream_hybrid_strain_to_vel = subsampled_vel_stream + sta_filt_one_comp

print(subsampled_filtered_stream_hybrid_strain_to_vel)

# Save the hybrid stream
subsampled_filtered_stream_hybrid_strain_to_vel.write(save_path_hybrid_strain_to_vel, format="MSEED")
